# NB10 — Repeated Validation and cycle_index Ablation

**Dissertation:** Explainable and Trustworthy Multimodal Deep Learning for Predictive Maintenance  
**Student:** 2023AA05069 | AIMLCZG628T | BITS Pilani  
**Notebook role:** Repeated validation (SPLIT_SEEDS = [21, 42, 84]) + cycle_index ablation  
**Environment:** Google Colab T4  
**Execution rule:** This notebook is executed in sections. The seed-42 reproduction gate (Section 7) must pass before seed 21 and seed 84 training begins.

---

## Frozen protocol

| Parameter | Value |
|---|---|
| Dataset | NASA C-MAPSS FD001 |
| RUL cap | 125 cycles |
| Rolling window (derived features) | 5 cycles |
| Sequence window | 30 cycles |
| Reference split seed | 42 |
| Model seed | 42 (fixed across all splits) |
| Split seeds | [21, 42, 84] |
| Test set access | None — test set not opened in this notebook |

**SPLIT_SEED** controls engine-cohort assignment.  
**MODEL_SEED** controls weight initialisation, dropout, and shuffling. It is fixed at 42 across all splits so that repeated validation measures engine-cohort sensitivity only, not mixed with weight-initialisation variation.

## Section 0 — Colab and Project Setup

Run this cell first. It mounts Drive, verifies the project root and GPU, and creates all output directories.

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Dissertation/Project/dissertation-rul-xai'
else:
    BASE = os.path.abspath('..')

# Verify project root
assert os.path.isdir(BASE), f'Project root not found: {BASE}'
assert os.path.isfile(f'{BASE}/data/processed/train_val_split_fd001.csv'), \
    'train_val_split_fd001.csv not found — check Drive path'

# Paths
RAW_DIR        = f'{BASE}/data/raw/CMAPSS'
PROCESSED_DIR  = f'{BASE}/data/processed'
ARRAY_DIR      = f'{BASE}/data/processed/multiview'
FV_BASE        = f'{BASE}/data/processed/final_validation'
MV_BASE        = f'{BASE}/models/final_validation'
RV_DIR         = f'{BASE}/reports/final_validation'

# Create final-phase output directories
for seed in [21, 42, 84]:
    os.makedirs(f'{FV_BASE}/split_seed_{seed}', exist_ok=True)
    os.makedirs(f'{MV_BASE}/split_seed_{seed}', exist_ok=True)
os.makedirs(RV_DIR, exist_ok=True)

# GPU check
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {len(gpus) > 0}')
if gpus:
    print(f'GPU device: {gpus[0]}')
else:
    print('WARNING: No GPU detected. DerivedOnlyMLP and MultiViewGRUFusion training will be slow.')

print(f'Project root: {BASE}')
print('Directories ready.')

## Section 1 — Environment Manifest

Record all package versions and hardware state before any computation. Saved to `reports/final_validation/experiment_environment.json`.

In [ ]:
import json
import platform
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb
import gc
from datetime import datetime

# Attempt deterministic ops (Colab T4 supports this from TF 2.8+)
deterministic_enabled = False
try:
    tf.config.experimental.enable_op_determinism()
    deterministic_enabled = True
except Exception as e:
    print(f'enable_op_determinism not available: {e}')

gpu_name = 'None'
if gpus:
    try:
        gpu_name = tf.test.gpu_device_name()
    except Exception:
        gpu_name = str(gpus[0])

env_manifest = {
    'recorded_at':         datetime.utcnow().isoformat() + 'Z',
    'python':              platform.python_version(),
    'tensorflow':          tf.__version__,
    'numpy':               np.__version__,
    'pandas':              pd.__version__,
    'scikit_learn':        sklearn.__version__,
    'xgboost':             xgb.__version__,
    'gpu_available':       len(gpus) > 0,
    'gpu_device':          gpu_name,
    'deterministic_ops':   deterministic_enabled,
    'platform':            platform.platform(),
}

env_path = f'{RV_DIR}/experiment_environment.json'
with open(env_path, 'w') as f:
    json.dump(env_manifest, f, indent=2)

print('Environment manifest:')
for k, v in env_manifest.items():
    print(f'  {k}: {v}')
print(f'Saved to: {env_path}')

## Section 2 — Frozen Experiment Configuration

All architectural and training parameters are fixed here. Nothing downstream modifies these values.

In [ ]:
# ── Seeds ──────────────────────────────────────────────────────────────────
SPLIT_SEEDS          = [21, 42, 84]   # controls engine-cohort assignment
REFERENCE_SPLIT_SEED = 42             # the mid-semester seed; gate must reproduce this
MODEL_SEED           = 42             # fixed; controls weight init, dropout, shuffling

# ── Dataset parameters ─────────────────────────────────────────────────────
RUL_CAP       = 125
WINDOW_SIZE   = 30
ROLLING_WIN   = 5
STRIDE        = 1
N_TRAIN_ENG   = 80
N_VAL_ENG     = 20

# ── Feature sets (verified against NB03 output) ────────────────────────────
SENSOR_COLS = [
    'sensor_measurement_11', 'sensor_measurement_4',  'sensor_measurement_12',
    'sensor_measurement_7',  'sensor_measurement_15', 'sensor_measurement_21',
    'sensor_measurement_20', 'sensor_measurement_2',  'sensor_measurement_17',
    'sensor_measurement_3',  'sensor_measurement_8',  'sensor_measurement_13',
    'sensor_measurement_9',  'sensor_measurement_14',
]  # 14 variable sensors (Feature Set B)

METADATA_COLS = ['unit_number', 'time_in_cycles', 'RUL', 'RUL_capped']
TARGET_COL    = 'RUL_capped'

# ── XGBoost configuration (frozen from NB04) ───────────────────────────────
XGB_PARAMS = dict(
    n_estimators    = 300,
    learning_rate   = 0.05,
    max_depth       = 4,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    objective       = 'reg:squarederror',
    random_state    = MODEL_SEED,
    n_jobs          = -1,
)

# ── GRU configuration (frozen from NB05) ──────────────────────────────────
GRU_MAX_EPOCHS = 30
GRU_BATCH      = 256
GRU_ES_PATIENCE= 10
GRU_LR_PATIENCE= 5
GRU_LR         = 0.001

# ── DerivedOnlyMLP / MultiViewGRUFusion configuration (frozen from NB06) ──
MLP_MAX_EPOCHS = 60
MLP_BATCH      = 128
MLP_ES_PATIENCE= 8
MLP_LR_PATIENCE= 4
MLP_MIN_LR     = 1e-5
MLP_LR         = 0.001

# ── Reference best epochs (seed 42, mid-semester) ─────────────────────────
REF_BEST_EPOCH = {
    'GRU':                22,
    'DerivedOnlyMLP':     60,   # hit maximum
    'MultiViewGRUFusion': 18,
}

# ── Reference metrics (seed 42, window-aligned validation) ────────────────
REF_METRICS = {
    'MultiViewGRUFusion': {'rmse': 12.0657, 'mae': 8.9406,  'r2': 0.9168},
    'XGBoost':            {'rmse': 12.4894, 'mae': 9.2675,  'r2': 0.9109},
    'DerivedOnlyMLP':     {'rmse': 13.1451, 'mae': 9.4204,  'r2': 0.9013},
    'GRU':                {'rmse': 13.1605, 'mae': 9.7182,  'r2': 0.9010},
}
METRIC_TOLERANCE = 0.005   # max acceptable absolute difference for RMSE gate

# ── Execution flags ────────────────────────────────────────────────────────
OVERWRITE_EXISTING       = False   # set True only to force rerun
RUN_CYCLE_INDEX_ABLATION = True

# ── Save frozen config ─────────────────────────────────────────────────────
frozen_config = {
    'SPLIT_SEEDS': SPLIT_SEEDS, 'REFERENCE_SPLIT_SEED': REFERENCE_SPLIT_SEED,
    'MODEL_SEED': MODEL_SEED, 'RUL_CAP': RUL_CAP, 'WINDOW_SIZE': WINDOW_SIZE,
    'ROLLING_WIN': ROLLING_WIN, 'STRIDE': STRIDE,
    'N_TRAIN_ENG': N_TRAIN_ENG, 'N_VAL_ENG': N_VAL_ENG,
    'SENSOR_COLS': SENSOR_COLS,
    'XGB_PARAMS': XGB_PARAMS,
    'GRU': {'max_epochs': GRU_MAX_EPOCHS, 'batch': GRU_BATCH,
            'es_patience': GRU_ES_PATIENCE, 'lr_patience': GRU_LR_PATIENCE, 'lr': GRU_LR},
    'MLP': {'max_epochs': MLP_MAX_EPOCHS, 'batch': MLP_BATCH,
            'es_patience': MLP_ES_PATIENCE, 'lr_patience': MLP_LR_PATIENCE,
            'min_lr': MLP_MIN_LR, 'lr': MLP_LR},
    'REF_BEST_EPOCH': REF_BEST_EPOCH,
    'REF_METRICS': REF_METRICS,
    'METRIC_TOLERANCE': METRIC_TOLERANCE,
    'OVERWRITE_EXISTING': OVERWRITE_EXISTING,
    'RUN_CYCLE_INDEX_ABLATION': RUN_CYCLE_INDEX_ABLATION,
}
config_path = f'{RV_DIR}/frozen_experiment_config.json'
with open(config_path, 'w') as f:
    json.dump(frozen_config, f, indent=2)
print(f'Frozen config saved to: {config_path}')
print(f'SPLIT_SEEDS: {SPLIT_SEEDS}  |  MODEL_SEED: {MODEL_SEED}  |  METRIC_TOLERANCE: {METRIC_TOLERANCE}')

## Section 3 — Raw Data Loading

Load raw FD001 training data from the original C-MAPSS files. Test data is not loaded in this notebook.

In [ ]:
INDEX_COLS  = ['unit_number', 'time_in_cycles']
OP_COLS     = [f'operational_setting_{i}' for i in range(1, 4)]
ALL_SENSOR_COLS = [f'sensor_measurement_{i}' for i in range(1, 22)]
ALL_COLS    = INDEX_COLS + OP_COLS + ALL_SENSOR_COLS

raw_train = pd.read_csv(
    f'{RAW_DIR}/train_FD001.txt', sep=r'\s+', header=None, names=ALL_COLS
)

# Compute and cap RUL
max_cycles = raw_train.groupby('unit_number')['time_in_cycles'].max().rename('max_cycle')
raw_train  = raw_train.join(max_cycles, on='unit_number')
raw_train['RUL']        = raw_train['max_cycle'] - raw_train['time_in_cycles']
raw_train['RUL_capped'] = raw_train['RUL'].clip(upper=RUL_CAP).astype(float)
raw_train.drop(columns=['max_cycle'], inplace=True)

all_units = sorted(raw_train['unit_number'].unique())
print(f'Raw training data loaded: {raw_train.shape}')
print(f'Total engines: {len(all_units)}')
print(f'RUL range (uncapped): [{raw_train["RUL"].min()}, {raw_train["RUL"].max()}]')
print(f'RUL range (capped):   [{raw_train["RUL_capped"].min()}, {raw_train["RUL_capped"].max()}]')

## Section 4 — Shared Preprocessing Functions

These functions implement the identical pipeline used in NB03 and NB06. They are used for all three splits.

In [ ]:
import random
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score


def make_engine_split(all_units, n_train, split_seed):
    """Reproducible engine-level 80/20 split using split_seed only.
    Uses legacy np.random API to match the NB03 split exactly."""
    np.random.seed(split_seed)
    shuffled    = np.random.permutation(all_units)
    train_units = sorted(shuffled[:n_train].tolist())
    val_units   = sorted(shuffled[n_train:].tolist())
    return train_units, val_units


def compute_derived_features(df, sensor_features, rolling_window=ROLLING_WIN):
    """Per-engine rolling mean, std, delta, and cycle_index. No cross-engine leakage."""
    parts = []
    for unit, unit_df in df.groupby('unit_number'):
        unit_df = unit_df.sort_values('time_in_cycles').copy()
        for s in sensor_features:
            unit_df[f'{s}_rmean'] = (
                unit_df[s].rolling(rolling_window, min_periods=1).mean()
            )
            unit_df[f'{s}_rstd'] = (
                unit_df[s].rolling(rolling_window, min_periods=1).std().fillna(0)
            )
            unit_df[f'{s}_delta'] = unit_df[s] - unit_df[s].iloc[0]
        unit_df['cycle_index'] = unit_df['time_in_cycles']
        parts.append(unit_df)
    return pd.concat(parts, ignore_index=True)


def fit_and_apply_scalers(train_df, val_df, feature_set_b, feature_set_c):
    """Fit scalers on training split only; apply to val. Returns scaled copies."""
    scaler_b = StandardScaler().fit(train_df[feature_set_b])
    scaler_c = StandardScaler().fit(train_df[feature_set_c])

    train_b = train_df.copy()
    val_b   = val_df.copy()
    train_c = train_df.copy()
    val_c   = val_df.copy()

    train_b[feature_set_b] = scaler_b.transform(train_df[feature_set_b])
    val_b[feature_set_b]   = scaler_b.transform(val_df[feature_set_b])
    train_c[feature_set_c] = scaler_c.transform(train_df[feature_set_c])
    val_c[feature_set_c]   = scaler_c.transform(val_df[feature_set_c])

    return train_b, val_b, train_c, val_c, scaler_b, scaler_c


def create_sequence_windows(df, sensor_cols, target_col, window_size=WINDOW_SIZE, stride=STRIDE):
    """Sliding 30-cycle windows of raw sensor sequence. Windows stay within engine."""
    X, y, meta = [], [], []
    for unit, unit_df in df.groupby('unit_number'):
        unit_df = unit_df.sort_values('time_in_cycles').reset_index(drop=True)
        feats   = unit_df[sensor_cols].values.astype(np.float32)
        targets = unit_df[target_col].values.astype(np.float32)
        cycles  = unit_df['time_in_cycles'].values
        raw_rul = unit_df['RUL'].values
        if len(unit_df) < window_size:
            continue
        for end in range(window_size - 1, len(unit_df), stride):
            start = end - window_size + 1
            X.append(feats[start:end + 1])
            y.append(targets[end])
            meta.append({'unit_number': unit, 'time_in_cycles': cycles[end],
                         'RUL': raw_rul[end], 'RUL_capped': targets[end]})
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32), pd.DataFrame(meta)


def create_multiview_windows(b_df, c_df, sensor_cols, derived_cols, target_col,
                              window_size=WINDOW_SIZE, stride=STRIDE):
    """Paired sequence + derived-feature windows. Both views share the same prediction cycle."""
    c_lookup = c_df.set_index(['unit_number', 'time_in_cycles'])
    X_seq, X_der, y, meta = [], [], [], []
    for unit, unit_df in b_df.groupby('unit_number'):
        unit_df = unit_df.sort_values('time_in_cycles').reset_index(drop=True)
        feats   = unit_df[sensor_cols].values.astype(np.float32)
        targets = unit_df[target_col].values.astype(np.float32)
        cycles  = unit_df['time_in_cycles'].values
        raw_rul = unit_df['RUL'].values
        if len(unit_df) < window_size:
            continue
        for end in range(window_size - 1, len(unit_df), stride):
            start = end - window_size + 1
            cycle = cycles[end]
            key   = (unit, cycle)
            if key not in c_lookup.index:
                continue
            X_seq.append(feats[start:end + 1])
            X_der.append(c_lookup.loc[key, derived_cols].values.astype(np.float32))
            y.append(targets[end])
            meta.append({'unit_number': unit, 'time_in_cycles': cycle,
                         'RUL': raw_rul[end], 'RUL_capped': targets[end]})
    return (np.array(X_seq, dtype=np.float32), np.array(X_der, dtype=np.float32),
            np.array(y, dtype=np.float32), pd.DataFrame(meta))


def evaluate_predictions(y_true, y_pred):
    """RMSE, MAE, R² with predictions clipped to [0, RUL_CAP]."""
    y_pred = np.clip(np.asarray(y_pred, dtype=np.float32), 0, RUL_CAP)
    return (
        round(float(root_mean_squared_error(y_true, y_pred)), 4),
        round(float(mean_absolute_error(y_true, y_pred)),     4),
        round(float(r2_score(y_true, y_pred)),                4),
    )


def compute_per_engine_metrics(y_true_arr, y_pred_arr, meta_df):
    """Per-engine RMSE and MAE from window-level predictions and metadata."""
    rows = []
    for unit in meta_df['unit_number'].unique():
        mask = (meta_df['unit_number'] == unit).values
        yt   = y_true_arr[mask]
        yp   = np.clip(y_pred_arr[mask], 0, RUL_CAP)
        rows.append({
            'unit_number': unit,
            'n_windows':   int(mask.sum()),
            'engine_rmse': round(float(root_mean_squared_error(yt, yp)), 4),
            'engine_mae':  round(float(mean_absolute_error(yt, yp)),     4),
            'mean_error':  round(float((yp - yt).mean()),                4),
        })
    return pd.DataFrame(rows).sort_values('unit_number').reset_index(drop=True)


print('Shared preprocessing functions defined.')

## Section 5 — Leakage Assertions

Checks applied to every split before training. Any failure raises an assertion error and stops execution.

In [ ]:
def run_leakage_assertions(train_units, val_units, train_c_df, val_c_df,
                            X_train_seq, X_val_seq, X_train_der, X_val_der,
                            y_train, y_val, train_meta, val_meta,
                            feature_set_b, derived_cols):

    # 1. Correct engine counts
    assert len(train_units) == N_TRAIN_ENG, \
        f'Expected {N_TRAIN_ENG} train engines, got {len(train_units)}'
    assert len(val_units) == N_VAL_ENG, \
        f'Expected {N_VAL_ENG} val engines, got {len(val_units)}'

    # 2. No overlap between train and val engines
    overlap = set(train_units) & set(val_units)
    assert len(overlap) == 0, f'Train/val engine overlap: {overlap}'

    # 3. All 100 engines accounted for
    assert len(set(train_units) | set(val_units)) == 100, \
        'Train + val engines do not sum to 100'

    # 4. Val engines in val data only
    train_eng_in_meta = set(train_meta['unit_number'].unique())
    val_eng_in_meta   = set(val_meta['unit_number'].unique())
    assert train_eng_in_meta == set(train_units), 'Train meta contains unexpected engines'
    assert val_eng_in_meta   == set(val_units),   'Val meta contains unexpected engines'
    assert len(train_eng_in_meta & val_eng_in_meta) == 0, \
        'Engine appears in both train and val metadata'

    # 5. normalized_cycle_age must not appear in model inputs
    all_input_cols = set(feature_set_b) | set(derived_cols)
    assert 'normalized_cycle_age' not in all_input_cols, \
        'normalized_cycle_age found in model inputs — leakage risk'

    # 6. cycle_index must be present
    assert 'cycle_index' in derived_cols, 'cycle_index missing from derived features'

    # 7. Target and metadata not in model inputs
    for forbidden in METADATA_COLS:
        assert forbidden not in all_input_cols, \
            f'Metadata/target column in model inputs: {forbidden}'

    # 8. Array shapes consistent
    assert X_train_seq.shape[0] == len(y_train) == len(train_meta)
    assert X_val_seq.shape[0]   == len(y_val)   == len(val_meta)
    assert X_train_seq.shape[1] == WINDOW_SIZE
    assert X_train_seq.shape[2] == len(feature_set_b)
    assert X_train_der.shape[1] == len(derived_cols)

    # 9. No NaN in arrays
    assert not np.isnan(X_train_seq).any(), 'NaN in X_train_seq'
    assert not np.isnan(X_train_der).any(), 'NaN in X_train_der'
    assert not np.isnan(X_val_seq).any(),   'NaN in X_val_seq'
    assert not np.isnan(X_val_der).any(),   'NaN in X_val_der'
    assert not np.isnan(y_train).any(),     'NaN in y_train'
    assert not np.isnan(y_val).any(),       'NaN in y_val'

    print('  All leakage assertions passed.')


print('Leakage assertion function defined.')

## Section 6 — Frozen Model Builders

Exact architectures from NB05 (GRU) and NB06 (DerivedOnlyMLP, MultiViewGRUFusion). Nothing changed.

In [ ]:
from tensorflow.keras import layers, models, callbacks
import joblib
from xgboost import XGBRegressor


def build_gru(window_size, n_sensors):
    model = models.Sequential([
        layers.Input(shape=(window_size, n_sensors)),
        layers.GRU(64, return_sequences=True),
        layers.GRU(32),
        layers.Dense(50, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1),
    ], name='GRU')
    model.compile(optimizer=tf.keras.optimizers.Adam(GRU_LR), loss='mse', metrics=['mae'])
    return model


def build_derived_mlp(n_features):
    inp = layers.Input(shape=(n_features,), name='degradation_feature_view')
    x   = layers.Dense(64, activation='relu')(inp)
    x   = layers.Dropout(0.2)(x)
    x   = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(1, name='rul_prediction')(x)
    model = models.Model(inputs=inp, outputs=out, name='DerivedOnlyMLP')
    model.compile(optimizer=tf.keras.optimizers.Adam(MLP_LR), loss='mse', metrics=['mae'])
    return model


def build_multiview_gru(window_size, n_sensors, n_derived):
    seq_in  = layers.Input(shape=(window_size, n_sensors), name='sensor_sequence_view')
    seq_x   = layers.GRU(64, return_sequences=False, name='sensor_gru_encoder')(seq_in)
    seq_x   = layers.Dropout(0.2)(seq_x)

    der_in  = layers.Input(shape=(n_derived,), name='degradation_feature_view')
    der_x   = layers.Dense(64, activation='relu', name='degradation_dense_1')(der_in)
    der_x   = layers.Dropout(0.2)(der_x)
    der_x   = layers.Dense(32, activation='relu', name='degradation_dense_2')(der_x)

    fused   = layers.Concatenate(name='view_fusion')([seq_x, der_x])
    z       = layers.Dense(64, activation='relu', name='fusion_dense_1')(fused)
    z       = layers.Dropout(0.2)(z)
    z       = layers.Dense(32, activation='relu', name='fusion_dense_2')(z)
    out     = layers.Dense(1, name='rul_prediction')(z)

    model = models.Model(inputs=[seq_in, der_in], outputs=out, name='MultiViewGRUFusion')
    model.compile(optimizer=tf.keras.optimizers.Adam(MLP_LR), loss='mse', metrics=['mae'])
    return model


def get_gru_callbacks():
    return [
        callbacks.EarlyStopping(
            monitor='val_loss', patience=GRU_ES_PATIENCE,
            restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=GRU_LR_PATIENCE,
            min_lr=1e-6, verbose=1),
    ]


def get_mlp_callbacks():
    return [
        callbacks.EarlyStopping(
            monitor='val_loss', patience=MLP_ES_PATIENCE,
            restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=MLP_LR_PATIENCE,
            min_lr=MLP_MIN_LR, verbose=1),
    ]


print('Model builder functions defined.')

## Section 7 — Seed-42 Reproduction Gate

**This gate must pass before seed 21 and seed 84 training begins.**

The gate:
1. Regenerates the seed-42 split from raw data using the shared preprocessing functions
2. Compares engine assignments against the saved `train_val_split_fd001.csv`
3. Compares array shapes and metadata against the saved NPZ files
4. Recalculates metrics from the existing saved prediction CSVs
5. Checks all four reference RMSE values within tolerance
6. Stops execution if any structural check fails

Minor floating-point differences (< METRIC_TOLERANCE) in metrics are acceptable. Structural or ordering differences are not.

In [ ]:
print('=== SEED-42 REPRODUCTION GATE ===')
print()

# ── Step 1: Regenerate split ───────────────────────────────────────────────
regen_train_units, regen_val_units = make_engine_split(
    all_units, N_TRAIN_ENG, REFERENCE_SPLIT_SEED
)

# ── Step 2: Load saved split assignment ───────────────────────────────────
saved_split = pd.read_csv(f'{PROCESSED_DIR}/train_val_split_fd001.csv')
saved_train_units = sorted(saved_split[saved_split['split'] == 'train']['unit_number'].tolist())
saved_val_units   = sorted(saved_split[saved_split['split'] == 'val']['unit_number'].tolist())

# ── Step 3: Compare engine assignments ────────────────────────────────────
train_match = (regen_train_units == saved_train_units)
val_match   = (regen_val_units   == saved_val_units)

print(f'Regenerated train engines: {len(regen_train_units)}')
print(f'Saved train engines:       {len(saved_train_units)}')
print(f'Train assignment match:    {train_match}')
print(f'Val assignment match:      {val_match}')

if not train_match or not val_match:
    mismatched_train = set(regen_train_units) ^ set(saved_train_units)
    mismatched_val   = set(regen_val_units)   ^ set(saved_val_units)
    print(f'Mismatched train engines: {mismatched_train}')
    print(f'Mismatched val engines:   {mismatched_val}')
    raise RuntimeError(
        'GATE FAILED: Engine assignment mismatch. '
        'The split function does not reproduce the mid-semester split. '
        'Investigate before proceeding.'
    )

print('Engine assignment check: PASS')

In [ ]:
# ── Step 4: Regenerate preprocessing for seed 42 ──────────────────────────
train_raw_42 = raw_train[raw_train['unit_number'].isin(regen_train_units)].copy()
val_raw_42   = raw_train[raw_train['unit_number'].isin(regen_val_units)].copy()

# Derived features
train_c_raw_42 = compute_derived_features(train_raw_42, SENSOR_COLS)
val_c_raw_42   = compute_derived_features(val_raw_42,   SENSOR_COLS)

# Feature lists
DERIVED_COLS = [
    c for c in train_c_raw_42.columns
    if c.endswith('_rmean') or c.endswith('_rstd') or c.endswith('_delta')
    or c == 'cycle_index'
]
FEATURE_SET_C = SENSOR_COLS + DERIVED_COLS

# Verify feature counts
assert len(SENSOR_COLS)  == 14, f'Expected 14 sensor cols, got {len(SENSOR_COLS)}'
assert len(DERIVED_COLS) == 43, f'Expected 43 derived cols, got {len(DERIVED_COLS)}'
assert 'cycle_index' in DERIVED_COLS
assert 'normalized_cycle_age' not in DERIVED_COLS

# Scale
train_b_42, val_b_42, train_c_42, val_c_42, scaler_b_42, scaler_c_42 = fit_and_apply_scalers(
    train_c_raw_42, val_c_raw_42, SENSOR_COLS, FEATURE_SET_C
)

print(f'Sensor cols (Feature Set B): {len(SENSOR_COLS)}')
print(f'Derived cols:                {len(DERIVED_COLS)}')
print(f'Feature Set C total:         {len(FEATURE_SET_C)}')
print(f'Train rows: {len(train_b_42)}  |  Val rows: {len(val_b_42)}')

In [ ]:
# ── Step 5: Build multi-view arrays and compare against saved NPZ ──────────
X_seq_42, X_der_42, y_42, meta_42 = create_multiview_windows(
    val_b_42, val_c_42, SENSOR_COLS, DERIVED_COLS, TARGET_COL
)
X_seq_train_42, X_der_train_42, y_train_42, train_meta_42 = create_multiview_windows(
    train_b_42, train_c_42, SENSOR_COLS, DERIVED_COLS, TARGET_COL
)

# Load saved NPZ
saved_npz = np.load(f'{ARRAY_DIR}/val_multiview_fd001_window30.npz')
saved_meta = pd.read_csv(f'{ARRAY_DIR}/val_multiview_meta_fd001_window30.csv')
saved_train_npz = np.load(f'{ARRAY_DIR}/train_multiview_fd001_window30.npz')

# Shape checks
shape_seq_match   = (X_seq_42.shape       == saved_npz['X_seq'].shape)
shape_der_match   = (X_der_42.shape       == saved_npz['X_derived'].shape)
shape_y_match     = (y_42.shape           == saved_npz['y'].shape)
shape_tr_seq_match= (X_seq_train_42.shape == saved_train_npz['X_seq'].shape)

print(f'Val X_seq  — regen: {X_seq_42.shape}   saved: {saved_npz["X_seq"].shape}   match: {shape_seq_match}')
print(f'Val X_der  — regen: {X_der_42.shape}   saved: {saved_npz["X_derived"].shape}   match: {shape_der_match}')
print(f'Val y      — regen: {y_42.shape}   saved: {saved_npz["y"].shape}   match: {shape_y_match}')
print(f'Train X_seq— regen: {X_seq_train_42.shape}   saved: {saved_train_npz["X_seq"].shape}   match: {shape_tr_seq_match}')

if not all([shape_seq_match, shape_der_match, shape_y_match, shape_tr_seq_match]):
    raise RuntimeError(
        'GATE FAILED: Array shape mismatch between regenerated and saved NPZ. '
        'Preprocessing functions may differ from NB03/NB06. Investigate.'
    )
print('Array shape check: PASS')

# Numerical comparison (val sequences)
seq_close = np.allclose(X_seq_42, saved_npz['X_seq'], atol=1e-5, rtol=1e-5)
der_close = np.allclose(X_der_42, saved_npz['X_derived'], atol=1e-5, rtol=1e-5)
y_close   = np.allclose(y_42,     saved_npz['y'],          atol=1e-5, rtol=1e-5)

print(f'Val X_seq numerical match (allclose): {seq_close}')
print(f'Val X_der numerical match (allclose): {der_close}')
print(f'Val y     numerical match (allclose): {y_close}')

if not seq_close:
    max_diff = float(np.abs(X_seq_42 - saved_npz['X_seq']).max())
    print(f'  Max absolute difference in X_seq: {max_diff:.6f}')
if not der_close:
    max_diff = float(np.abs(X_der_42 - saved_npz['X_derived']).max())
    print(f'  Max absolute difference in X_der: {max_diff:.6f}')

# Metadata check
meta_rows_match  = (len(meta_42) == len(saved_meta))
meta_units_match = (sorted(meta_42['unit_number'].unique()) ==
                    sorted(saved_meta['unit_number'].unique()))
print(f'Val meta rows  — regen: {len(meta_42)}   saved: {len(saved_meta)}   match: {meta_rows_match}')
print(f'Val meta units — regen: {sorted(meta_42["unit_number"].unique())} match: {meta_units_match}')

In [ ]:
# ── Step 6: Recalculate metrics from saved prediction CSVs ─────────────────
# Load saved window-aligned prediction files
pred_fusion  = pd.read_csv(f'{BASE}/reports/predictions/val_predictions_MultiViewGRUFusion_window30_fd001.csv')
pred_gru     = pd.read_csv(f'{BASE}/reports/predictions/val_predictions_GRU_B_window30_fd001.csv')
pred_derived = pd.read_csv(f'{BASE}/reports/predictions/val_predictions_DerivedOnlyMLP_window30_fd001.csv')
pred_xgb_all = pd.read_csv(f'{BASE}/reports/predictions/val_predictions_XGBoost_C_fd001.csv')

# XGBoost prediction file contains all 4291 val rows; filter to window-aligned 3711
# Window-aligned rows: those present in the windowed val metadata
window_keys  = pred_fusion[['unit_number', 'time_in_cycles']]
pred_xgb_aligned = pred_xgb_all.merge(window_keys, on=['unit_number', 'time_in_cycles'], how='inner')

print(f'Fusion predictions:          {len(pred_fusion)} rows')
print(f'GRU predictions:             {len(pred_gru)} rows')
print(f'DerivedOnlyMLP predictions:  {len(pred_derived)} rows')
print(f'XGBoost all-val rows:        {len(pred_xgb_all)}')
print(f'XGBoost window-aligned rows: {len(pred_xgb_aligned)}')

assert len(pred_xgb_aligned) == 3711, \
    f'Expected 3711 window-aligned XGBoost rows, got {len(pred_xgb_aligned)}'

# Recalculate metrics
rmse_fusion,   mae_fusion,   r2_fusion   = evaluate_predictions(
    pred_fusion['RUL_capped'].values, pred_fusion['prediction'].values)
rmse_gru,      mae_gru,      r2_gru      = evaluate_predictions(
    pred_gru['RUL_capped'].values,    pred_gru['prediction'].values)
rmse_derived,  mae_derived,  r2_derived  = evaluate_predictions(
    pred_derived['RUL_capped'].values, pred_derived['prediction'].values)
rmse_xgb,      mae_xgb,      r2_xgb      = evaluate_predictions(
    pred_xgb_aligned['RUL_capped'].values, pred_xgb_aligned['prediction'].values)

recalc = {
    'MultiViewGRUFusion': {'rmse': rmse_fusion, 'mae': mae_fusion, 'r2': r2_fusion},
    'GRU':                {'rmse': rmse_gru,    'mae': mae_gru,    'r2': r2_gru},
    'DerivedOnlyMLP':     {'rmse': rmse_derived,'mae': mae_derived,'r2': r2_derived},
    'XGBoost':            {'rmse': rmse_xgb,    'mae': mae_xgb,    'r2': r2_xgb},
}

print()
print('Recalculated metrics from saved prediction CSVs:')
for model, m in recalc.items():
    ref = REF_METRICS[model]
    diff = abs(m['rmse'] - ref['rmse'])
    status = 'PASS' if diff <= METRIC_TOLERANCE else 'FAIL'
    print(f'  {model:<25} RMSE {m["rmse"]:.4f}  ref {ref["rmse"]:.4f}  diff {diff:.4f}  [{status}]')

In [ ]:
# ── Step 7: Run leakage assertions for seed-42 ────────────────────────────
print('Running leakage assertions for seed-42 split...')
run_leakage_assertions(
    regen_train_units, regen_val_units,
    train_c_42, val_c_42,
    X_seq_train_42, X_seq_42,
    X_der_train_42, X_der_42,
    y_train_42, y_42,
    train_meta_42, meta_42,
    SENSOR_COLS, DERIVED_COLS,
)
print()

# ── Step 8: Build and print gate summary table ────────────────────────────
gate_rows = [
    ('Training engines',      N_TRAIN_ENG,           len(regen_train_units),
     'PASS' if len(regen_train_units) == N_TRAIN_ENG else 'FAIL'),
    ('Validation engines',    N_VAL_ENG,             len(regen_val_units),
     'PASS' if len(regen_val_units) == N_VAL_ENG else 'FAIL'),
    ('Training windows',      14020,                 len(y_train_42),
     'PASS' if len(y_train_42) == 14020 else 'FAIL'),
    ('Validation windows',    3711,                  len(y_42),
     'PASS' if len(y_42) == 3711 else 'FAIL'),
    ('Seq shape (per sample)','(30, 14)',             str(X_seq_42.shape[1:]),
     'PASS' if X_seq_42.shape[1:] == (30, 14) else 'FAIL'),
    ('Derived feature count', 43,                    X_der_42.shape[1],
     'PASS' if X_der_42.shape[1] == 43 else 'FAIL'),
    ('Engine assignment',     'match',               'match' if train_match and val_match else 'MISMATCH',
     'PASS' if train_match and val_match else 'FAIL'),
    ('Array numerical match', 'allclose',            'allclose' if (seq_close and der_close and y_close) else 'DIFFERS',
     'PASS' if (seq_close and der_close and y_close) else 'WARN'),
    ('Fusion RMSE',           REF_METRICS['MultiViewGRUFusion']['rmse'], rmse_fusion,
     'PASS' if abs(rmse_fusion - REF_METRICS['MultiViewGRUFusion']['rmse']) <= METRIC_TOLERANCE else 'FAIL'),
    ('XGBoost aligned RMSE',  REF_METRICS['XGBoost']['rmse'],           rmse_xgb,
     'PASS' if abs(rmse_xgb - REF_METRICS['XGBoost']['rmse']) <= METRIC_TOLERANCE else 'FAIL'),
    ('DerivedOnlyMLP RMSE',   REF_METRICS['DerivedOnlyMLP']['rmse'],    rmse_derived,
     'PASS' if abs(rmse_derived - REF_METRICS['DerivedOnlyMLP']['rmse']) <= METRIC_TOLERANCE else 'FAIL'),
    ('GRU RMSE',              REF_METRICS['GRU']['rmse'],               rmse_gru,
     'PASS' if abs(rmse_gru - REF_METRICS['GRU']['rmse']) <= METRIC_TOLERANCE else 'FAIL'),
]

gate_df = pd.DataFrame(gate_rows, columns=['Check', 'Expected', 'Reproduced', 'Status'])
print('\n=== SEED-42 REPRODUCTION GATE SUMMARY ===')
print(gate_df.to_string(index=False))

all_pass = all(r == 'PASS' for r in gate_df['Status'])
warn_only = all(r in ('PASS', 'WARN') for r in gate_df['Status'])

print()
if all_pass:
    print('GATE RESULT: ALL CHECKS PASSED — seed 21 and seed 84 training is authorised.')
elif warn_only:
    print('GATE RESULT: PASSED WITH WARNINGS — review WARN items above before proceeding.')
    print('If numerical differences are due to platform floating-point, document and proceed.')
    print('If differences are structural, investigate before proceeding.')
else:
    failed = gate_df[gate_df['Status'] == 'FAIL']['Check'].tolist()
    raise RuntimeError(
        f'GATE FAILED on: {failed}. '
        'Do not proceed to seed 21/84 training until failures are resolved.'
    )

In [ ]:
# ── Step 9: Save gate artefacts ────────────────────────────────────────────
gate_csv_path = f'{RV_DIR}/seed42_reproduction_gate.csv'
gate_df.to_csv(gate_csv_path, index=False)
print(f'Gate table saved: {gate_csv_path}')

gate_manifest = {
    'gate_run_at':         datetime.utcnow().isoformat() + 'Z',
    'reference_split_seed': REFERENCE_SPLIT_SEED,
    'model_seed':           MODEL_SEED,
    'metric_tolerance':     METRIC_TOLERANCE,
    'all_pass':             bool(all_pass),
    'warn_only':            bool(warn_only),
    'train_assignment_match': bool(train_match),
    'val_assignment_match':   bool(val_match),
    'array_seq_close':        bool(seq_close),
    'array_der_close':        bool(der_close),
    'array_y_close':          bool(y_close),
    'recalculated_metrics':   recalc,
    'reference_metrics':      REF_METRICS,
    'n_train_windows':        int(len(y_train_42)),
    'n_val_windows':          int(len(y_42)),
    'n_train_engines':        int(len(regen_train_units)),
    'n_val_engines':          int(len(regen_val_units)),
    'derived_col_count':      int(len(DERIVED_COLS)),
    'sensor_col_count':       int(len(SENSOR_COLS)),
    'gate_checks':            gate_df.to_dict(orient='records'),
}

manifest_path = f'{RV_DIR}/seed42_reproduction_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(gate_manifest, f, indent=2)
print(f'Gate manifest saved: {manifest_path}')

print()
print('=== SECTION 7 COMPLETE ===')
print('Review gate output above. Share seed42_reproduction_gate.csv before starting seed 21/84.')